# 07 — Workflows: When Agents Need Choreography

**What you'll learn:**
- When to use a Workflow vs an Agent
- Core concepts: Executors, Edges, WorkflowContext
- Building a loan application processing pipeline

---

## Agent vs Workflow

| Use an **Agent** when... | Use a **Workflow** when... |
|--------------------------|---------------------------|
| The task is open-ended or conversational | The process has well-defined steps |
| You need autonomous tool use and planning | You need explicit control over execution order |
| A single LLM call (possibly with tools) suffices | Multiple agents or functions must coordinate |

**Rule of thumb:** If you can draw it as a flowchart, it's a Workflow.

## Core Concepts

- **Executor**: A processing unit — can be AI agents or plain Python functions
- **Edge**: A connection between executors that routes messages
- **WorkflowContext**: Provides `send_message()` and `yield_output()` methods
- **WorkflowBuilder**: Wires executors and edges into a runnable graph

In [1]:
from dotenv import load_dotenv

from agent_framework import (
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    handler,
)
from typing import Never

load_dotenv()

# Note: This notebook uses pure workflows (no LLM) to teach the concepts.
# Notebook 08 combines agents + workflows.

True

## Scenario: Loan Application Pipeline

A bank receives a loan application and processes it through three stages:

```
Application → [Validate] → [Credit Check] → [Decision Engine] → Result
```

Each stage is an **Executor**. Let's define the data model and executors.

In [2]:
from dataclasses import dataclass


@dataclass
class LoanApplication:
    """Raw loan application from the customer."""
    applicant_name: str
    annual_income: float
    loan_amount: float
    loan_purpose: str
    employment_years: int


@dataclass
class ValidatedApplication:
    """Application after validation — augmented with validation status."""
    application: LoanApplication
    is_valid: bool
    issues: list[str]


@dataclass
class CreditReport:
    """Result of the credit check stage."""
    application: LoanApplication
    credit_score: int
    debt_to_income_ratio: float
    existing_loans: int


@dataclass
class LoanDecision:
    """Final underwriting decision."""
    applicant_name: str
    decision: str  # approved, denied, referred
    reason: str
    approved_amount: float | None = None
    interest_rate: float | None = None


print("Data models defined")

Data models defined


## Step 1: Class-Based Executor — Validate Application

Class-based executors inherit from `Executor` and use the `@handler`
decorator to mark the processing method. The handler's type annotations
define what messages the executor accepts.

In [3]:
class ValidateApplication(Executor):
    """Checks completeness and basic eligibility of a loan application."""

    @handler
    async def validate(
        self, app: LoanApplication, ctx: WorkflowContext[ValidatedApplication]
    ) -> None:
        issues = []

        if app.annual_income <= 0:
            issues.append("Annual income must be positive")
        if app.loan_amount <= 0:
            issues.append("Loan amount must be positive")
        if app.loan_amount > app.annual_income * 5:
            issues.append(f"Loan amount (${app.loan_amount:,.0f}) exceeds 5x annual income")
        if app.employment_years < 1:
            issues.append("Minimum 1 year of employment required")

        is_valid = len(issues) == 0
        status = "PASSED" if is_valid else "FAILED"
        print(f"  [Validate] {app.applicant_name}: {status} ({len(issues)} issues)")

        await ctx.send_message(
            ValidatedApplication(application=app, is_valid=is_valid, issues=issues)
        )


print("ValidateApplication executor defined")

ValidateApplication executor defined


## Step 2: Function-Based Executor — Credit Check

For simpler executors, use the `@executor` decorator on a plain function.

In [4]:
import random


@executor(id="credit_check")
async def credit_check(
    validated: ValidatedApplication, ctx: WorkflowContext[CreditReport | LoanDecision]
) -> None:
    """Simulates a credit bureau lookup."""
    if not validated.is_valid:
        # Fast-reject: skip credit check for invalid applications
        print(f"  [Credit Check] Skipping — application invalid: {validated.issues}")
        decision = LoanDecision(
            applicant_name=validated.application.applicant_name,
            decision="denied",
            reason=f"Application validation failed: {'; '.join(validated.issues)}",
        )
        await ctx.send_message(decision)
        return

    # Simulate credit score based on income and employment
    app = validated.application
    base_score = 650
    income_bonus = min(int(app.annual_income / 10000) * 5, 100)
    employment_bonus = min(app.employment_years * 10, 50)
    score = base_score + income_bonus + employment_bonus + random.randint(-30, 30)
    score = max(300, min(850, score))  # clamp to valid range

    dti = round((app.loan_amount * 0.06) / app.annual_income, 2)  # simplified DTI

    print(f"  [Credit Check] {app.applicant_name}: score={score}, DTI={dti:.0%}")

    await ctx.send_message(
        CreditReport(
            application=app,
            credit_score=score,
            debt_to_income_ratio=dti,
            existing_loans=random.randint(0, 3),
        )
    )


print("credit_check executor defined")

credit_check executor defined


## Step 3: Decision Engine Executor

Takes the credit report and makes an approve / deny / refer decision.

In [5]:
@executor(id="decision_engine")
async def decision_engine(
    report: CreditReport, ctx: WorkflowContext[Never, LoanDecision]
) -> None:
    """Makes the final lending decision based on credit report."""
    app = report.application

    if report.credit_score >= 720 and report.debt_to_income_ratio < 0.36:
        decision = "approved"
        rate = 5.25
        reason = f"Excellent credit ({report.credit_score}) and low DTI ({report.debt_to_income_ratio:.0%})"
        amount = app.loan_amount
    elif report.credit_score >= 640:
        decision = "approved"
        rate = 6.75
        amount = min(app.loan_amount, app.annual_income * 3)
        reason = f"Good credit ({report.credit_score}), approved at reduced amount"
    elif report.credit_score >= 580:
        decision = "referred"
        rate = None
        amount = None
        reason = f"Marginal credit ({report.credit_score}) — requires manual review"
    else:
        decision = "denied"
        rate = None
        amount = None
        reason = f"Credit score ({report.credit_score}) below minimum threshold of 580"

    result = LoanDecision(
        applicant_name=app.applicant_name,
        decision=decision,
        reason=reason,
        approved_amount=amount,
        interest_rate=rate,
    )
    print(f"  [Decision] {app.applicant_name}: {decision.upper()}")

    # yield_output sends the result back to the workflow caller
    await ctx.yield_output(result)


print("decision_engine executor defined")

decision_engine executor defined


## Step 4: Build and Run the Workflow

Wire the executors together with edges using `WorkflowBuilder`.

In [6]:
# Build the workflow graph
validate = ValidateApplication(id="validate")

workflow = (
    WorkflowBuilder(start_executor=validate)
    .add_chain([validate, credit_check, decision_engine])
    .build()
)

print("Workflow built: Validate → Credit Check → Decision")

Workflow built: Validate → Credit Check → Decision


## Demo: Processing Loan Applications

Let's run two applications through the pipeline.

In [7]:
# Application 1: Strong applicant
app1 = LoanApplication(
    applicant_name="Jennifer Torres",
    annual_income=95_000,
    loan_amount=280_000,
    loan_purpose="Home purchase",
    employment_years=8,
)

print("=== Processing: Jennifer Torres ===")
run_result = await workflow.run(app1)

for output in run_result.get_outputs():
    decision: LoanDecision = output
    print(f"\n  Result: {decision.decision.upper()}")
    print(f"  Reason: {decision.reason}")
    if decision.approved_amount:
        print(f"  Approved Amount: ${decision.approved_amount:,.0f}")
        print(f"  Interest Rate: {decision.interest_rate}%")

=== Processing: Jennifer Torres ===
  [Validate] Jennifer Torres: PASSED (0 issues)
  [Credit Check] Jennifer Torres: score=758, DTI=18%
  [Decision] Jennifer Torres: APPROVED

  Result: APPROVED
  Reason: Excellent credit (758) and low DTI (18%)
  Approved Amount: $280,000
  Interest Rate: 5.25%


In [8]:
# Application 2: Weak applicant — will fail validation
app2 = LoanApplication(
    applicant_name="Test User",
    annual_income=30_000,
    loan_amount=500_000,  # way over 5x income
    loan_purpose="Investment property",
    employment_years=0,   # less than 1 year
)

print("\n=== Processing: Test User ===")
run_result = await workflow.run(app2)

for output in run_result.get_outputs():
    decision: LoanDecision = output
    print(f"\n  Result: {decision.decision.upper()}")
    print(f"  Reason: {decision.reason}")


=== Processing: Test User ===
  [Validate] Test User: FAILED (2 issues)
  [Credit Check] Skipping — application invalid: ['Loan amount ($500,000) exceeds 5x annual income', 'Minimum 1 year of employment required']


## What Just Happened?

```
LoanApplication
    │
    ▼
┌──────────────────┐
│ ValidateApplication│  ── checks income, amount, employment
└────────┬─────────┘
         │ ValidatedApplication
         ▼
┌──────────────────┐
│   credit_check   │  ── simulates credit bureau lookup
└────────┬─────────┘     (or fast-rejects if validation failed)
         │ CreditReport
         ▼
┌──────────────────┐
│ decision_engine   │  ── approve / deny / refer
└────────┬─────────┘
         │ LoanDecision (yield_output)
         ▼
    Workflow caller receives result
```

Key points:
- Each executor is **type-safe** — it declares what it accepts and produces
- Messages flow through edges automatically
- `yield_output()` sends results back to the caller
- No LLM was needed — workflows can be purely deterministic

---

## Key Takeaways

- **Executors** process messages — use class-based (`Executor + @handler`) or function-based (`@executor`)
- **Edges** connect executors into a directed graph
- `ctx.send_message()` forwards output to the next executor
- `ctx.yield_output()` sends results back to the workflow caller
- Workflows are **deterministic** — the execution path is explicitly defined, not decided by an LLM
- Workflows can include **conditional routing** (the credit_check executor skips ahead on validation failure)

**Next up:** [08 — Multi-Agent Orchestration](./08-multi-agent-orchestration.ipynb) —
combine agents with workflows for sequential claims processing and concurrent policy quoting.